# Stage 1: Setup, data, and the first model check

This notebook prepares the experiment. It does **not** run the research baseline,
select layers, or apply interventions.

1. In Colab, choose **Runtime > Change runtime type > GPU**.
2. Run the cells below in order. You will upload `unlearning-stage1.zip` from the project `dist` folder.
3. Have a Hugging Face read token ready in Colab Secrets under `HF_TOKEN`, or use the hidden prompt.
   Never paste the token into a code cell or chat.
4. Your account must have access to
   [Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct).
5. Download the results in the last cell, even if the model check fails.

Expected time: setup and the first model download may take several minutes.
A successful one-question check is evidence that the pipeline runs, not that the model performs well.


## 1. Upload the project source

Select `unlearning-stage1.zip`. This contains the code, configuration, tests, and instructions.
It contains no credentials, model weights, or saved notebook outputs.


In [ ]:
import io
import os
from pathlib import Path
import zipfile
from google.colab import files

uploaded = files.upload()
archive_name = 'unlearning-stage1.zip'
if archive_name not in uploaded:
    raise ValueError('Please select unlearning-stage1.zip from the project dist folder.')
project = Path('/content/unlearning_task')
project.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as archive:
    for entry in archive.infolist():
        target = (project / entry.filename).resolve()
        if project.resolve() not in target.parents:
            raise ValueError('Unexpected path inside the archive.')
    archive.extractall(project)
del uploaded
os.chdir(project)
print('Project ready:', project)


## 2. Install project dependencies

Colab supplies the GPU-compatible PyTorch installation. We record its exact version in the results.
Other direct dependencies are pinned, with separate versions for Python 3.13 and older runtimes.
Installation errors stop this cell. This does not change your local computer.
The experiment commands run in fresh Python processes, so they see the installed package versions.


In [ ]:
import os
import subprocess
import sys
os.chdir('/content/unlearning_task')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--only-binary=:all:', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)


## 3. Check the code and runtime

The data tests use invented questions and need no downloads. The tiny model has random weights;
it checks that our forward-pass code runs. It is not the research model.


In [ ]:
import json
from pathlib import Path
import subprocess
import sys

def run(*args, required=True):
    result = subprocess.run([sys.executable, *args])
    if required and result.returncode:
        raise RuntimeError('The command failed. Read the error above before continuing.')
    return result.returncode

run('-m', 'unittest', 'discover', '-s', 'tests', '-v')
run('-m', 'unlearning', 'doctor')
run('-m', 'unlearning', 'smoke', '--tiny', '--output', 'outputs/stage1/tiny_smoke.json')


## 4. Prepare and verify the real data

Download ten small public data files. The model is not involved in this step.
The full profile reserves 128 localization, 128 development, and 256 final-test questions per main dataset,
plus 64 general-biology control questions. Duplicate question text is removed before sampling.

A **manifest** is a record of the selected data, settings, source versions, and file checksums.
Keep it with the experiment. The final-test questions are reserved for later evaluation;
do not inspect their answers to choose a method.


In [ ]:
run('-m', 'unlearning', 'prepare', '--profile', 'full', '--output', 'data/prepared/full')
run('-m', 'unlearning', 'verify', '--data', 'data/prepared/full')
manifest = json.loads(Path('data/prepared/full/manifest.json').read_text())
print('Data fingerprint:', manifest['request_hash'])


## 5. Provide model access privately

Having a Hugging Face account is separate from having access to this model.
Check the [model page](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) while logged in.
If needed, request access there first. A token alone does not bypass that requirement.

This cell tries the Colab secret `HF_TOKEN`; if it is absent, it uses a hidden input prompt.
It does not print the token or save it in the notebook, Git, or result files.
It then checks the token and downloads one small model configuration file to test access.
Rerunning this cell reloads the secret, so a corrected token replaces a previously invalid one.


In [ ]:
import getpass
import os
from google.colab import userdata

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
except Exception:
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face read token (hidden): ').strip()
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('No token supplied. Add a Colab secret or rerun this cell.')
run('-m', 'unlearning', 'access-check', required=False)
access_report = json.loads(Path('outputs/stage1/model_access.json').read_text())
if access_report['status'] != 'passed':
    print('Fix the reported access issue before running the model cell. You can still download the reports below.')


## 6. Load the recommended model and run one harmless question

This downloads the pinned Llama-3.2-1B-Instruct weights and uses the GPU.
It checks the four answer-label tokens and runs one forward pass on a simple arithmetic question.
We use float32 initially to keep later gradient checks straightforward.

If this fails, keep the failure report and download the results below. Do not start the main experiments.


In [ ]:
access_report = json.loads(Path('outputs/stage1/model_access.json').read_text())
if access_report['status'] != 'passed':
    raise RuntimeError('Model access has not passed. Follow the message from the previous cell first.')
model_check_exit = run('-m', 'unlearning', 'smoke', required=False)
if model_check_exit:
    failure = json.loads(Path('outputs/stage1/model_smoke.json').read_text())
    print(failure.get('error', 'Read model_smoke.json for details.'))
    print('Stage 1 is not complete: the recommended-model check needs attention.')
else:
    print('Stage 1 checks passed. Stop here so we can review before Stage 2.')


## 7. Download the evidence

Download this ZIP before leaving Colab. It includes the prepared data and manifest, runtime information,
and model-check reports. It excludes credentials, caches, and model weights.
Bring `model_smoke.json` back for review; no screenshots of credentials are needed.


In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

evidence = Path('/content/stage1-results.zip')
with zipfile.ZipFile(evidence, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in (Path('data/prepared'), Path('outputs/stage1')):
        if folder.exists():
            for path in sorted(folder.rglob('*')):
                if path.is_file() and path.suffix in ('.json', '.jsonl'):
                    archive.write(path, path.as_posix())
    archive.write('configs/experiment.json', 'configs/experiment.json')
files.download(str(evidence))
